# 01b · Retrain: more movement practice, 2 epochs (QLoRA, Unsloth)

Same as notebook 01, with a data mix aimed at the Layout A/B weak spots: **250 movement examples (was 60)**, 200 pass examples (was 400), all 125 impulse grabs, and **2 epochs**. Upload the same `impulse_v3_dataset.csv`.

**Run:** `Runtime → Change runtime type → A100 GPU`, then `Runtime → Run all`. You upload one file: `impulse_v3_dataset.csv` (a file picker opens at step 2; or drag it into the Files panel first).
**Output:** a ZIP downloads at the end (`simply_qwen25_7b_impulse_v3_adapter_*.zip`). You upload it into notebooks 02 and 03.
**Time:** about 100 minutes (install ≈ 4, load ≈ 3, untuned check ≈ 4, train ≈ 75, tuned check ≈ 4, save ≈ 2).

What it does: verifies the uploaded impulse_v3 CSV (hashes, JSON, split leakage) → measures token lengths (no truncation) →
scores the **untuned** model on a fixed validation subset → memory check on the longest example → QLoRA fine-tune
(4-bit NF4, LoRA r=16 on all attention + MLP layers, answer-only loss, validation loss every 50 steps, best checkpoint kept)
→ scores the **tuned** model on the same rows → saves the adapter, checks it against memory → downloads one ZIP.
No paid API calls. Whole-trip results come from notebooks 02/03.

Switches (step 3): `MODE = "smoke"` trains 3 steps only, to test the pipeline (its adapter is rejected by 02/03);
`FULL_DATA = True` uses all 1,753 rows (~80 min of training); `RUN_FINAL_TEST = True` also scores the held-out test split.

In [ ]:
import subprocess, time
SMI = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", SMI or "none")
assert SMI, "No GPU. Runtime → Change runtime type → A100 GPU."
assert int(SMI.split(",")[1].split()[0]) >= 38000, "Training needs a 40 GB A100 for 16K-token observations."
STAMP = time.strftime("%Y%m%d_%H%M%S")

### 1 · Install

In [ ]:
# Same install the original working notebook used: plain pip, output shown live.
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "unsloth", "datasets", "peft", "accelerate", "bitsandbytes",
    "sentencepiece", "protobuf",
])
print("Install finished. If Colab shows a 'Restart session' popup, click Restart, then Runtime -> Run all.")

### 2 · Unpack the scoring code, then provide the dataset CSV

In [ ]:
#@title Unpack the bundled scoring code and dataset fingerprint (hash-verified; do not edit)
import base64, hashlib, io, shutil, sys, zipfile
from pathlib import Path
BUNDLE_SHA256 = "e56d7f161cbab259cdbb10ae6400aa33fd3d65338718468463e3079853f5e4e6"
BUNDLE_B64 = "UEsDBBQAAAAIAAAAcV0AnxjmdRUAAHBHAAAlAAAAc2ltdWxhdGlvbi9ldmFsdWF0aW9uL2ltcHVsc2VfZXZhbC5wed08a3PbRpLf9SvmvHULIIZpyY+qLXrpKyfx7qbKcby2slUumgdB5JDECgQYDCiZq+L99uvHPAFQUrz36fwhIubR3dPT75nJo0ePftpsd6WSTy53e9HsSqlSUS+XZVFJsZDzQhV1JdS8bopqJfJqIdqm2IpGbuumVWJZN6JdS/H3G1mJv7+rP74RN3VzNTo5+U5cFAw5m9dVK7+2F6JQNFgBqBKhL4uqaBF+vRQ3a4CQ48/lk7JQrYgf6fmPErFq8kuYfSJEvt029bYp8laOxE8tEJIvlPjl/bvPYpnPW0awbOp/AbSLogL6NnJxIWDSZtsKta5veMimXsgS4MVb2ai6ygVSKB6Lix92TSMrGNoCioskFaqmCdfPYEg+X8smUqLML2WpRN5IUcq8qfLLUo5wycgomWn+XRDfpGJkQKra1pWC73yVFxUsUQMUDVCVCqDEcpyAIacz5jSA2m02eVPQdDHPq7oq5jkA3VViWZTIDOaWaOs217R92Ldr4C4SAUTXNxUs+AKIy5DB2Q6Yry5eweoKIhH2XlTyGqgAJpcFIILNLapr4AaAm+NGqdHJo0ePTmBjgCbxT2Cc+d3IE+D6BnCVpeSxQnf9UO9g/5sUNzzfle2imLcnJ+cf3745z358++HNx/Of374//yQmIo4+Vfn8SkWpiL7Pr2Szp19IU76S1PwX2lr89QE4A4tciJ8lrDdKTv729s2787997sH80MDa5hLn/JgXzV78UbxdrXDGrx//CqNo0LpYrXHEvAGJBMZC7x/EOe7781B4rFSDvmTQc5lfFmXR7i9EjNpxAaO2WxDwTLX7EgQIOCgCmRoR2LYAPgPfc0Cz3FVzowbtOm+FB3YsACuIdoO6sau2ZV5VsOailRuxyfdiVY/Ep3rXzKX44dM/xEbmlRoLHtaI/zkdnT1LAcWm+Aqz4PPZi1TwCopriQ0vn49Ozn96+zH7/pdf3/9IHIsjOwJYcjp6cQpqEEcEgxqecYPGQk1np0kihPiDAMUAgrFhLKpa45ICuKVOPp1/fvc2+/ntm/fZh4+/fP/m+5/e/XT+GVDeWlhjQSQLjQ0/kWSPImx6+fxAyOqq3JMN2jbyid0pRQvCpdyCwItw8m00z7fwl3DkX2ELizmYKBRyaP3T6ekh5UmGADPhbHDCCzfBLeHuKc9e4pQDcOPN+VsW0apus7zN1FqWS5TCXbOCwfgLmOlLGYlojlYBLQfuBszNlkXbosiBWVgUC5Axmok6viwaRXBkWawKsFIg1ycnoIgkgBmwLvagJ2NaSSPbXVOBLfjaxnGVbySxGH+k4hKUeYEy7ctMsfRFVrye8DAQkvd1JQ3G+lLJ5jpHSY95o0J8aE5GZQ32XHePFJihNo4C9RkLWM5ZMj2bme4v1Q/rugYZA1wwRm7HPOJ0ZjCDnQCFBaBoKUPcYM4+aA/A/gP2CqzVQlzuQW1VW29kM7rcFeUi054iY0u8F08Ft1/KdX5d1E1eZouiQeN3LdUI7SQiWIODgj1+wIpOZyc0AykGz7iIt3kLdrPSlOK/Td7O1wCtkSMFfme+NmNSwpPYgbAjIFM83k0nZufgQ8Q/8nIn3zYN7P8ysgwoZLkAU6HQQYMUa9iHyMHVW0VwR6um3m3js4TJZvZNhraZ+skeQj+NG60ksCG0lVGCDodW3kSfaHT85eZxkmr8vogFYI5YZD2tI5xgc1EoHVNCsMMmakoEznxpvbUAeBmgTQWaC03/53pHLlj3idF3/yW+xPH0v5PZ4+RL8mUUJakHgdY/Zh5ZayeD1QBoUI420FcYinoMfVad+4N8RItCto5I/BJAUzojLnvjcvDizWrvhuqG4dEL0JkrleXlvF7XJUyKFjWEDyiB1CVMDzWB8UBh9ebDZ9muYR6oXqwRchtIwCJ5+jIKFsHmVMlKga++Zt54M6lbeN1dIAdtFjLWxmvUhdAUgehD4EKOBX8AxTwKBTSKEqPGr0BkCzIvrYEJplj5lgYijzYVusHZnB+R80+F4epTwyGInzagdoWWRwqpMPrUkCCKu4b4EOy4aPOVGolfq6sK4jqUamQt6nBrTQ8OSTUO4AZIt16xhsbag0IA5izD0ciiwUEWCIwwWhVpmrNnZ9typyJkEwKhLCFGcvSyp10BmQly2aYTZXImJhMQhLzE0GvctTd/gQhPetrME42QzsR/TNAPVjJyuN2ycc92ldpCaA0WbkF0Br0DAP0hd5FDGjTprsVQyr1A27Vc5VXUg6PbDeOGpuHeFENzL+u6jIndf4R4gwGlwZRD0gO4lWqefzvENIQwgIA3sL9QTAEx58kgI7iyZsBKi0P9BFD7m6VxaDjnzU4an87imS0wC2g3GFvF5BP0dnntGAge2EfBblOQhdgVJCdywZOmUQX+9HJPIRi472iWiiu5n5T55nKRCwipFW4u+OsKMomZJxIcfRItBBShTyPdAnBC7+tRBR681RmR0TU7D10JUKCBuUmA2WeHB01zBUK2JldZUUEs2LQBQzAelBiK3GLyEKKiVVBOAUvQDLHOGSdGs4OPGEKgmKH8tsurFk3wIAwkArVpKQZwaiFgukyodrlbgLhlqoQ8kCPmYBFg2j7KDUgSViN4rMiXLebQEsMObKZsXVtJ9gVzCea0AumiOgMkWGAuVVuUJcS4ciEX1mIyEOKSWyJG8JgrB2H8THyn19QYeoZZYLsz7FCdHewN0us3SJ5YkjR/OjUVZs6Ag4kptEyFTQlUohNPXXNoIb9WnFRqyTBSLL8CTMirqCJDYEi1bQYx4njPTzDGQeXG4gRQ8K1go9A1UR9zR1FF5zInR1bXr2jPQsqgxYotzyJUItbukDLyfK/0FtIQxWMSrlPBOJN92kIWuFcMTLDWg/wfEYoPkotYNoFMg7SV4jjwBvUNoGFhw9FMmCKPXUrk9c0atkDAGuZXNayYkrj5nutWsBBIi0E8NnJR7DYjsdoVQHZW5dcZ4L2ScouFHcrriC2Wm1T7okSX4SPdKivlshWvxdlLWutr8fz0P0dm7+mvY/DESJltypx5ss4cO61h1hP0ojNMqhR7bRxg4fTtfCeRnTJ8JDYFhm2K1lHD6/C0J4W04qsxx1xvymgOdp1ZX2NMi2ZzptnMTltXdIBQYtCfJ8gh8/VUkwCto+enfeJt1q3JphrNhDLdTxQGWTdPQffMkmSKOWFiYcD2U/ipZfugvcYEGkFOqYwwG/CmQQVAQwsd3nH3SINzSEryEsYNFc38GEuH5jOk6YWQqBO9yh2nd2ixU2OTJ8cNeWp2MDR1BMRZK4Cg19VxsAPCR3Y59Rjg1j+N+pY7Dbyw3te++w2SaTuCMl/pya12jyhkHgFajzSbtd48JDcw/9C+F9VOhkSgM3utZaNbWJo9BEAgv2TWXW2MLInBQfsmHpv9fPpUvPg2+FxGI9gM84nG8WcP9stvg20qbhz1Hwf/rLM7kC7wzmCN6B7UTiRHOAfLMrxzSUdiRxhPxl7QuB2LeDssgNuOqFlLgmLiQA7Z18Fi3zQoTcS+a0aoebVHSo7KLqlYoFwJq7pXNvRDCR2LkAnO2FLHGI1o1faiEO4UWPLHPUADmfoVoieQVhcLCE0+YZkRmd/AFJHzIQ2E5CWaVuCKhBThylXV2mbvmKORTPwKIpLDXJVf5xJihfh8v+WKV+pVv/q5HZMYJJykzApPbDD612tKaVFUsoJAXjcmKFq3EfpLSpbcHsMXWGygLzo8FCV439hDSwUIyAnaJqEN44IEGhlCPaJvFSf3LwnJS4UjDjjHMKZMOQio+fallObWN9XvSSI4AeDcgbYaI2vAcTzIMBzAzxEsvWnVTdGu42hTX8vMrw0YePwXTYCuOZhlkc0JkZWQ1ZYZgVpISAEqkkKDFqSdEZN1oYjoLnQeqgJrRgPs8HEPhV+HLiDtOIjNAzSB6tZ4jPFgqo4BMuHTnaAG+EmZOgHlKFuvbVsrOsnlKKwbos1lWToeK9nH6YRTy6xRMsLHRsJkQCqzR5jIUquPvu0xKqTP1RqjJFx2RiEnVe3uca++YwKPiT4N2T5oV8lfdaRhMglo8PoSe07ikWAjHi7sYX93S44bdM2pTY3eLrvCsmiYJ6aQZEGw2zpu8bcxAMdExAagpouDSR3wC8PQO1IIK0N3sM96QwsYZz3Yid0FeSA6N3IU+ey3FXiTOUYBPewcSRVNLbkBxxhXu42EHzUddlf1Bu2LdTLmWM2MgXzEG0RlNO+TUOC2awTBzYKYbwzYGwXO42LHWICXzsEr0cUCEd9GkPQqOkEfi+lO4Vl8rhRVs9rZIXnlAI1Fk9/o+wroQp3HBfJKWRFiUh366OCnVXbPl6JfQHTNSLrkADBQrhv5265oIDg0gg8rXHQCfxia8tUM2Nh/FdsjC9eqSqk7S3KG65jg/Klb/QzivSnVIqmWlva6z/xuV+Ib1J3ho650+LAxDWITS6HLMYwRMwbxaGxlZ/QKPDDrnuqQO0fEDc5MkWNyn0115rqoy1wHWySbpqNYhjDDoJocD9aDtHeLXVwZHlR3gvEuSgISjCCnNhSsGFTHvbJj3OFurBDBQ8yd1U0G+t6YCp05kfEMCsmvyRJuA5gRGmHQvgeY5NQdB9qchxtm7rivkw2lIbLw2NCOHTpVnBE6zCLGWqJCUBBuQpibKZDHTY51PvaonlsCAJ4eZCTAMJD+doBRZTHLV42UVM0esyswsTNlcQ+JRBP0T3HoslLrwoKRveXI7d0EhPgRUQdPCFAXLY3c40lkV5d4vtElvBLiawoJhffdgc99zkk9iGLthY3X7En30dik007h5L/rcjsLspqFMmJ+o2ATl/AkW7MrMkYe28CY6ZMg1Ej2PaRu3LiRKKbK3vgx0oaVWXR6VXDW7kk0uW0YwO4ba/7NtCPzvEy8CWdUHPaw8s+je+I/CLWvJPdD1grDtVBfbAPIXbW6H25PDTrrD3rvB3e5z7R9uz242wudzwxP17Rhuz2Yk3j8L4LH+ewaKKpK/cAsHQjHUi8I9bwGnoVDANH0SEYfBEsjMkng8Zdz8VqApnYpsykNoLtpKKoocgAc1bdvxR66GwzAgEoOJwFqu2BQtDnEzUhnCLprDPo701umx8aOJt71b4AdHiD4qEyUbkfaZG+QeDJaA6KVPGAFbp8fvIAh+i2YgPzaFMwfIjN9MdQHBVt9YAijuw6BrajdPNdssWkamKClAYXu9f8AXHUfZRrPw8BtG71wDMFxm+0mt9sU2fAYFoBXSDptVUdYHJ7AYmtcWt0M5tR5WdaOAePaX1tvAWkA1DdfvPajsO9lzXHIll8UhlneRVpHxpqL3ozlGbTGz8R3bjz81sx+KmLX+li3JnwHybaTfPB4yiHj09Epx+cs6jax9AmlxV+rbAt54XHT1ucyKMU3c+gwKBPoJkyJ37+2wWds9tbGbWNPAvB+D1/+6eI93OkZDEOAN/EANAoysXHAS/RondJA8hahEJPJHw4Zv0WYOx5kMFj8/YI86Jforgzdwgy5rnORb2e2B2MyEd7NymMs1qP1Jcz/V0zGxyy1wvcad3mgDsxZqDg227DASHcQk23p2F9855PZ6JvH43FGrN9q4EJcoO4twtHLlHmDkjtxMK+GgwIfyAAqzTX3HRRKNb7UsMxcb3dvZ/CO9NpVyMw7GbSTq13eLJq8KIW51F0v6Rp5/4GNiPk1zZN8B3/w3Qk/tElshYwua+YrvttFuYN5AcMlIRM0uMFY0hvl8zkyHW8u6eF0TNgZBWRkZW2H4LOnDNr4Kg7SOLFtvF7qwHu2WO0zq5vgUFQ/OpZJ9Rd3miN4femdkiks4OKCKB2kO3CmWwuFMrd58URa33VW5sCnpMs/AIZgjO89M5r3L51RP01PxZzGGPJGdOEntnpUzKkChvUeJpbvHFCqW9CdPZeuGoLxJiMwdScNxfJaR4PeU6UYSUmsSaQhpKLIO57gn71zy5T++NybmcIQ9ST6FLPYdgqe5r54UZlN8/nqnXmu+J6D7vMx+RVR0Bg6ruMXXfgLa1hNsVhIFpl/yjnfCjx15U5/lXo5BL1zw4ArrDSAzzSgIbNJexKMNcqLNzMCZYhhUio0ty7lEp+qzXrHBhTieGehNIlOQl2Z3EfH9wUYo76Xoy/6UtuIjsHs+U/QFR6fkQgPrp50pXtZIOD644k4C3rsObXHM27jpw862vEW4Q1cgpXaNfqRhNcOGQ1eatPnWXqfoz6tdqt7ZFENNe4fkKXHD6wQVxzp5HyAdQN1XE/uBijoVoyJXVpmg+HG2TnR12wIXjV60ueK3LGdi/Voozf4BlXhRaoZy9IpyVq730o7PkEZKkAf+sKG59tmQ42cTJ/gMQJdFKS8096BDUYxsKnTVroKhsYgLrr3gAv/ni3bsYJL2A0tpjh+s9agc3gUWqGB+r1hlFfa7VfxB7hGp+zgzX8jurJU/EZ2A1dDcjJ4IkAS2X+JxJ7iMOaV8eU4sag5SQX/Xs3pBmRtXrYe3XV/oUgc+4fptljgxV0mFT5CYu8UCwuajLat8oePf7SnsgV8662MaHSfPHXLycMPf/pgjj156pWnO6+DfgekxKVCHaCWN+MhflF6+9tO4p1ivS/euG5X7+hi18zXuRqa6rqWuxJCstID4jhnpg2X080okg37IpO+wmdXXL0eCxQZJy2ZlZbZndIyHTiRIacQHg0oPp2hntQa7/4Y3dGB6SwqJkl8oAuDXTPtBFt9HEHPwMfWEXSglXW9Ba/c6sfbjvFsX7vdVJWhG7w7FUqC1zo7JO5JY75aNXKFsT+9G/RchCnpc3vXclOA1E47G8yRo74RCtM69q07yd/v41OtcGqr0Q7Ia2eyI5cfYk0EV2Ao5NB22Cyke/fgNiJbQucTwrstwxfKuWrksqU2ELMOFWFBfkCTjM70yjrttKdzA9AtG7pYgl3BJ7IZrinAYJZPRKLaMKesMz0CkTI4tc6bMMUuxGtx6lzgNwJneTDkgrSiCBN8Fp9EPBVnp6fwX4ba31QtZscQHVd4w/QjA/rMDwEfNRIa7pH++8AeMSYa6GDvfSAHbI9Vqk5PD9ThxM/xbyNMf8nDmtRNN6DlL/M9qk3QzW0YftGgTq3XDmTbxm2JPgnfaWbq3ItbfBBVfl2suKixrctivu+AgzBlWawgML49JNzSnxEcH+L/3oKqwc5EkmHoHLHZwzP64Y+eto5/nFNiCDtQZkt6QRj9Gyz1tWGpzwE/JIeQMB1k3OLfh5DlbnBT1XSYqKGar5t7F0U6U/K2McigtO/Slpf+epOtgaD/pwY+ZdahpisQ+SGniI1N1ivEhzqvBN4Y0c1U3MELzadJdDj5X1BLAwQUAAAACAAAAHFdzyfRaEIBAABAAgAAHwAAAHRyYWluaW5nX2RhdGEvZGF0YXNldF9tZXRhLmpzb25NkctuwyAQRff+Cot1FsDw7LafUVUWDGPFqh2nfkRto/x7IVVDNujM1eUyM1yblqWwhZU29tKydZjO43eXz31cqdso4JGW7gLskI3TnGjsFvrcad3W7OdF7Zf5h04dztM03EO4NzYgcFCOlBCmx6RQWo+WvBXkLErjs4c93V7nfUHq1mOQ2pQQFMk4oD4JoSA5Fa1VNkDyMReJB9/HBL1CVCZ4sNILq6NDJwwPOkZzD6evkOeh0uq1aVu2LWE45SJ74VCESxiHPP4wF1V6eRe3PF0pnWra218Mjnui1EU64XEKy0dudD6faSnJb+XOK+dcssM/qoq2onugEBWrV/KKUPHJoCuaiv6BUL2qhinFmva9TILr5WnHTmlyKXKIRruQF07WlEcAtfBkU/4j0JIc7y0Gb4Rx3noN2vkeFHeeNbdfUEsBAhQDFAAAAAgAAABxXQCfGOZ1FQAAcEcAACUAAAAAAAAAAAAAAIABAAAAAHNpbXVsYXRpb24vZXZhbHVhdGlvbi9pbXB1bHNlX2V2YWwucHlQSwECFAMUAAAACAAAAHFdzyfRaEIBAABAAgAAHwAAAAAAAAAAAAAAgAG4FQAAdHJhaW5pbmdfZGF0YS9kYXRhc2V0X21ldGEuanNvblBLBQYAAAAAAgACAKAAAAA3FwAAAAA="
_blob = base64.b64decode(BUNDLE_B64)
assert hashlib.sha256(_blob).hexdigest() == BUNDLE_SHA256, "Embedded bundle is corrupted"
PROJECT = Path("/content/simply")
shutil.rmtree(PROJECT, ignore_errors=True); PROJECT.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(_blob)) as _z:
    _z.extractall(PROJECT)
del _blob, BUNDLE_B64
sys.path.insert(0, str(PROJECT / "simulation"))
print("Scoring code ready in", PROJECT)

In [ ]:
#@title Provide the dataset CSV (impulse_v3_dataset.csv)
# If you already dragged the CSV into the Colab Files panel (/content), it is used; otherwise a file picker opens.
import glob, os, shutil
from pathlib import Path
found = sorted(glob.glob("/content/**/impulse_v3_dataset*.csv", recursive=True))
if not found:
    from google.colab import files
    uploaded = files.upload()
    found = [name for name in uploaded if name.endswith(".csv")]
    assert found, "No CSV uploaded. Re-run this cell and choose impulse_v3_dataset.csv."
    found = [str(Path("/content") / found[0])]
CSV_PATH = Path(found[0])
print("Using", CSV_PATH, f"({CSV_PATH.stat().st_size / 2**20:.0f} MB)")

### 3 · Verify the data, choose settings and training rows

In [ ]:
import csv, hashlib, json, sys, shutil
from collections import Counter
from pathlib import Path
import evaluation.impulse_eval as ie

MODE = "pilot"          # "smoke" = 3 optimizer steps to test the pipeline; its adapter is rejected by notebooks 02/03
FULL_DATA = False       # True = all 1,753 rows (~80 min of training)
RUN_FINAL_TEST = False  # True = also score the held-out test split once (~12 min)
assert MODE in ("pilot", "smoke")

DATA = PROJECT / "training_data"
manifest = json.loads((DATA / "dataset_meta.json").read_text())
assert manifest["dataset"] == "simply_impulse_teacher_v3" and manifest["model_requests"] == 0
SOURCE_SHA256 = manifest["frozen_source_sha256"]       # ties this adapter to the exact prompt/code used in notebooks 02/03
assert hashlib.sha256(CSV_PATH.read_bytes()).hexdigest() == manifest["csv_sha256"], "dataset CSV changed or is the wrong file"
csv.field_size_limit(sys.maxsize)
data, provenance = {"train": [], "validation": [], "test": []}, {}
with open(CSV_PATH, newline="", encoding="utf-8") as handle:
    for row in csv.DictReader(handle):
        split = row["split"]
        data[split].append({"messages": [{"role": "user", "content": row["prompt"]}, {"role": "assistant", "content": row["response"]}]})
        provenance[split, len(data[split])] = row               # label, base_id, prompt_sha256, target_sha256 per row
assert {k: len(v) for k, v in data.items()} == manifest["examples"], "row counts differ from the expected dataset"

bases, prompts = {}, {}
for split, rows in data.items():                          # schema, allowed steps, hashes, split leakage
    bases[split], prompts[split] = set(), set()
    for line, row in enumerate(rows, 1):
        meta = provenance[split, line]
        assert [m["role"] for m in row["messages"]] == ["user", "assistant"]
        prompt, answer = (m["content"] for m in row["messages"])
        action = json.loads(answer)
        assert set(action) == {"step", "product_id", "reason"} and all(isinstance(v, str) for v in action.values())
        assert action["step"] in ie.observation(prompt)["allowed_steps"]
        assert hashlib.sha256(prompt.encode()).hexdigest() == meta["prompt_sha256"]
        assert hashlib.sha256(answer.encode()).hexdigest() == meta["target_sha256"]
        bases[split].add(meta["base_id"]); prompts[split].add(meta["prompt_sha256"])
for a, b in (("train", "validation"), ("train", "test"), ("validation", "test")):
    assert not bases[a] & bases[b] and not prompts[a] & prompts[b], "split leakage"
label_of = lambda split, i: provenance[split, i + 1]["label"]
print("Data checks passed:", {s: len(r) for s, r in data.items()}, "| benchmark shoppers excluded:", len(manifest["excluded_benchmark_shoppers"]))

CAPS = {"impulse_pass": 200, "list_grab": 120, "checkout": 50, "move": 250}   # training subset; impulse_grab: keep all
seen, keep = Counter(), []
for i in range(len(data["train"])):
    label = label_of("train", i)
    if FULL_DATA or seen[label] < CAPS.get(label, 10**9):
        seen[label] += 1; keep.append(i)
print(f"Training on {len(keep)} of {len(data['train'])} rows:", dict(seen))

EVAL_QUOTA = {"impulse_grab": 999, "impulse_pass": 24, "list_grab": 10, "checkout": 8, "move": 4}   # fixed validation subset
taken, eval_idx = Counter(), []
for i in range(len(data["validation"])):
    label = label_of("validation", i)
    if taken[label] < EVAL_QUOTA[label]:
        taken[label] += 1; eval_idx.append(i)
eval_rows = [data["validation"][i] for i in eval_idx]
print("Validation scoring subset:", dict(taken))

OUT = Path("/content/impulse_v3_adapter"); shutil.rmtree(OUT, ignore_errors=True); (OUT / "adapter").mkdir(parents=True)

### 4 · Load the 4-bit model, tokenize, and score the untuned model

In [ ]:
from unsloth import FastLanguageModel        # import before transformers
import gc, importlib.metadata, inspect, math, statistics, time
import torch
from datasets import Dataset
from huggingface_hub import HfApi

MODEL_ID, MAX_SEQ_LENGTH = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit", 16384
REVISION = HfApi().model_info(MODEL_ID).sha
model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_ID, revision=REVISION,
    max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True)      # 4-bit NF4 base, frozen
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def ids(messages, generation_prompt):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=generation_prompt)
    return tokenizer(text, add_special_tokens=False)["input_ids"]

def tokenize(row):                                # loss only on the assistant's JSON answer
    full, prefix = ids(row["messages"], False), ids(row["messages"][:-1], True)
    assert full[:len(prefix)] == prefix, "chat template prefix mismatch"
    labels = [-100] * len(prefix) + full[len(prefix):]
    assert any(l != -100 for l in labels), "no answer tokens to learn from"
    return {"input_ids": full, "attention_mask": [1] * len(full), "labels": labels}

rows = [tokenize(data["train"][i]) for i in keep]
eval_tokenized = [tokenize(data["validation"][i]) for i in eval_idx[:48]]
lengths = [len(r["input_ids"]) for r in rows]
length_summary = {"rows": len(rows), "median": statistics.median(lengths), "max": max(lengths)}
print("Token lengths:", length_summary, "(never truncated)")
assert max(lengths) + 256 <= MAX_SEQ_LENGTH, "example too long for the context window"
print("Learned text of the first example:", tokenizer.decode([t for t in rows[0]["labels"] if t != -100]))
train_dataset, eval_dataset = Dataset.from_list(rows), Dataset.from_list(eval_tokenized)

def generate_responses(rows_to_score, label):
    # Greedy, no repetition penalty: exactly how notebooks 02/03 serve the model.
    FastLanguageModel.for_inference(model); model.eval()
    out, started = [], time.time()
    for n, row in enumerate(rows_to_score, 1):
        text = tokenizer.apply_chat_template(row["messages"][:-1], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
        with torch.inference_mode():
            gen = model.generate(**inputs, max_new_tokens=256, do_sample=False, temperature=None, top_p=None, top_k=None,
                                 repetition_penalty=1.0, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
        out.append(tokenizer.decode(gen[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip())
        if n % 25 == 0:
            print(f"  {label}: {n}/{len(rows_to_score)} ({time.time() - started:.0f}s)", flush=True)
        del inputs, gen
    return out

def offline_eval(rows_to_score, label):
    metrics, records = ie.score_offline(rows_to_score, generate_responses(rows_to_score, label))
    (OUT / f"offline_eval_{label}.json").write_text(json.dumps({"metrics": metrics, "records": records}, indent=2))
    keys = ("strict_schema_rate", "observation_valid_rate", "exact_action_agreement", "list_grab_recall", "checkout_recall",
            "off_list_proposals", "rule_violation_rate")
    print(label, json.dumps({k: metrics[k] for k in keys}), "\nimpulse:", json.dumps(metrics["impulse"]),
          "\nby tier:", json.dumps(metrics["impulse_by_tier"]))
    return metrics

baseline = offline_eval(eval_rows, "validation_untuned")

### 5 · Memory check, then train (QLoRA)

In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments, set_seed
SEED = 3407
FastLanguageModel.for_training(model)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0, bias="none", random_state=SEED,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth")
model.print_trainable_parameters()
assert all("lora_" in n for n, p in model.named_parameters() if p.requires_grad), "base weights must stay frozen"
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, pad_to_multiple_of=8, label_pad_token_id=-100, return_tensors="pt")

# Memory check on the longest example (one forward/backward, no update) before committing ~40 minutes.
longest = max(rows, key=lambda r: len(r["input_ids"]))
batch = {k: v.to(model.device) for k, v in collator([longest]).items()}
model.train(); model.zero_grad(set_to_none=True)
with torch.autocast(device_type="cuda", dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16):
    loss = model(**batch).loss
loss.backward(); model.zero_grad(set_to_none=True); del batch, loss; gc.collect(); torch.cuda.empty_cache()
print("Longest example fits:", len(longest["input_ids"]), "tokens | peak", round(torch.cuda.max_memory_allocated() / 2**30, 1), "GiB")

set_seed(SEED)
smoke, bf16 = MODE == "smoke", torch.cuda.is_bf16_supported()
args = TrainingArguments(output_dir="/content/ckpt", per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=4, num_train_epochs=2, max_steps=3 if smoke else -1, learning_rate=1e-4, warmup_steps=0.05,   # transformers 5: a float below 1 is a ratio of total steps
    weight_decay=0.01, lr_scheduler_type="linear", optim="adamw_8bit", bf16=bf16, fp16=not bf16, logging_steps=10,
    eval_strategy="no" if smoke else "steps", eval_steps=50, prediction_loss_only=True,
    save_strategy="no" if smoke else "steps", save_steps=50, save_total_limit=2,
    load_best_model_at_end=not smoke, metric_for_best_model="eval_loss", greater_is_better=False,
    report_to="none", seed=SEED, data_seed=SEED, dataloader_num_workers=0, remove_unused_columns=False)
options = dict(model=model, args=args, train_dataset=train_dataset, eval_dataset=eval_dataset, data_collator=collator)
options["processing_class" if "processing_class" in inspect.signature(Trainer.__init__).parameters else "tokenizer"] = tokenizer
trainer = Trainer(**options)
result = trainer.train()
assert math.isfinite(result.metrics["train_loss"]), "loss is not finite"
print(result.metrics)

### 6 · Score the tuned model, save, verify and download the adapter

In [ ]:
import zipfile
from safetensors.torch import load_file
from google.colab import files

tuned = offline_eval(eval_rows, "validation_tuned")
print("\nBefore (untuned) -> after (tuned), same validation rows")
for key in ("strict_schema_rate", "observation_valid_rate", "exact_action_agreement", "list_grab_recall", "checkout_recall", "rule_violation_rate"):
    print(f"  {key}: {baseline[key]} -> {tuned[key]}")
for key in ("model_grab_rate", "teacher_grab_rate", "precision", "recall", "f1"):
    print(f"  impulse {key}: {baseline['impulse'][key]} -> {tuned['impulse'][key]}")
test_metrics = offline_eval(data["test"], "test_tuned") if RUN_FINAL_TEST else None

model.save_pretrained(str(OUT / "adapter")); tokenizer.save_pretrained(str(OUT / "adapter"))
adapter_sha = hashlib.sha256((OUT / "adapter" / "adapter_model.safetensors").read_bytes()).hexdigest()
saved, live = load_file(str(OUT / "adapter" / "adapter_model.safetensors")), dict(model.named_parameters())
mismatch = sum(1 for name, tensor in saved.items()
               if (q := live.get(name.replace(".lora_A.", ".lora_A.default.").replace(".lora_B.", ".lora_B.default."))) is None
               or not torch.equal(q.detach().cpu().to(tensor.dtype), tensor))
print(f"Saved {len(saved)} LoRA tensors, sha256 {adapter_sha[:16]}..., mismatches vs trained model: {mismatch}")
assert saved and mismatch == 0, "saved adapter differs from the trained model"

packages = {}
for name in ("unsloth", "unsloth_zoo", "transformers", "peft", "bitsandbytes", "torch", "datasets", "accelerate"):
    try: packages[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: packages[name] = "unavailable"
experiment = {"project": "Simply", "training_mode": "impulse_v3" if MODE == "pilot" else "impulse_v3_smoke",
    "method": "QLoRA_SFT", "base_model": MODEL_ID, "base_revision": REVISION, "dataset": manifest["dataset"],
    "dataset_csv_sha256": manifest["csv_sha256"], "frozen_source_sha256": SOURCE_SHA256, "frozen_commit": manifest["frozen_commit"],
    "rows_trained": len(keep), "full_data": FULL_DATA, "lora_rank": 16, "lora_alpha": 32, "learning_rate": 1e-4, "epochs": 2, "variant": "v3b_more_movement",
    "gradient_accumulation_steps": 4, "seed": SEED, "max_seq_length": MAX_SEQ_LENGTH, "token_lengths": length_summary,
    "training_arguments": args.to_dict(), "train_metrics": result.metrics, "assistant_output_only_loss": True, "load_in_4bit": True,
    "chat_template_sha256": hashlib.sha256(tokenizer.chat_template.encode()).hexdigest(), "adapter_sha256": adapter_sha,
    "lora_tensors": len(saved), "packages": packages, "gpu": torch.cuda.get_device_name(0),
    "offline_eval": {"validation_untuned": baseline, "validation_tuned": tuned, "test_tuned": test_metrics}}
(OUT / "experiment.json").write_text(json.dumps(experiment, indent=2, default=str))
(OUT / "training_log.json").write_text(json.dumps(trainer.state.log_history, indent=2))
ZIP = Path(f"/content/simply_qwen25_7b_impulse_v3_adapter_{STAMP}.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.rglob("*")):
        if path.is_file(): archive.write(path, str(path.relative_to(OUT)))
print("Created", ZIP, f"({ZIP.stat().st_size / 2**20:.0f} MiB)")
print("\n=== PASTE THIS INTO THE SIMPLY CHAT ===\n" + json.dumps({"mode": MODE, "adapter_sha256": adapter_sha[:16],
    "rows_trained": len(keep), "train_loss": result.metrics["train_loss"],
    "validation_untuned_impulse": baseline["impulse"], "validation_tuned_impulse": tuned["impulse"],
    "validation_tuned_by_tier": tuned["impulse_by_tier"], "tuned_rule_violation_rate": tuned["rule_violation_rate"],
    "tuned_exact_action_agreement": tuned["exact_action_agreement"],
    "test_tuned_impulse": test_metrics["impulse"] if test_metrics else None, "packages": packages}, indent=1, default=str))
files.download(str(ZIP))